# W4 — Milestone 1: Standard Autoencoder (KKBox)

Unsupervised AE trained **only on non-churned clients** to learn a compact
representation of the 59 tabular features from W1.  
Bottleneck sweep: `z_dim ∈ {8, 16, 32}`.

## 0. Setup

In [ ]:
import sys
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Notebook lives at workshops/w4/; Jupyter sets CWD to notebook dir.
_w4_root = Path().resolve()
_w4_src  = _w4_root / 'src'
_w1_root = _w4_root.parent / 'w1'
for _p in (str(_w4_src), str(_w1_root)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from constants import SEED, Z_DIMS, CHECKPOINTS_DIR, RUNS_DIR, FIGURES_DIR
from autoencoder.data_loader    import load_w1_data, filter_normal
from autoencoder.preprocessor   import TabularPreprocessor
from autoencoder.autoencoder_ae  import AEConfig, Autoencoder
from train.train_ae    import set_seed, build_loader, AETrainer, save_checkpoint, load_checkpoint
from evaluate.evaluate_ae import ReconstructionEvaluator

RUNS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(SEED)
print(f'device: {DEVICE}  |  z_dims: {Z_DIMS}')

## 1. Load Data

In [ ]:
df, y_all, features, categorical, numeric, split = load_w1_data()
fit_normal, stop_normal = filter_normal(split, y_all)

print(f'\nfit_normal   (AE train): {len(fit_normal):>9,}')
print(f'stop_normal  (AE val):   {len(stop_normal):>9,}')
print(f'test (full):             {len(split["test"]):>9,}  '
      f'(churn rate: {y_all[split["test"]].mean():.4f})')

## 2. Preprocessor

Fitted exclusively on `fit_normal` (non-churned February clients).
Constant numeric columns are detected and excluded from `x_num`.

In [ ]:
prep = TabularPreprocessor(numeric, categorical)
prep.fit(df.iloc[fit_normal])

print(f'numeric_dim (non-const): {prep.numeric_dim}')
print(f'constant cols excluded:  {prep.n_const}')
print(f'cat_cardinalities:       {prep.cat_cardinalities}')
print(f'input_dim (to model):    {prep.input_dim}')

x_num_fit,  x_cat_fit  = prep.transform(df.iloc[fit_normal])
x_num_stop, x_cat_stop = prep.transform(df.iloc[stop_normal])
x_num_test, x_cat_test = prep.transform(df.iloc[split['test']])
y_test = y_all[split['test']]

print(f'x_num_fit  shape: {x_num_fit.shape}')
print(f'x_num_test shape: {x_num_test.shape}')

## 3. z_dim Sweep

Train one AE per bottleneck size. Saves checkpoint + history CSV per run.

In [ ]:
results: dict = {}

for z in Z_DIMS:
    print(f'\n' + '='*55)
    print(f'  Training AE  z_dim={z}')
    print('='*55)

    config = AEConfig(
        numeric_dim=prep.numeric_dim,
        cat_cardinalities=prep.cat_cardinalities,
        z_dim=z,
        seed=SEED,
    )
    set_seed(SEED)
    model = Autoencoder(config).to(DEVICE)

    train_loader = build_loader(x_num_fit,  x_cat_fit,  config.batch_size, shuffle=True)
    val_loader   = build_loader(x_num_stop, x_cat_stop, config.batch_size, shuffle=False)

    trainer = AETrainer(model, config, DEVICE)
    history = trainer.fit(train_loader, val_loader)

    ckpt_path = CHECKPOINTS_DIR / f'ae_z{z}.pt'
    save_checkpoint(ckpt_path, model, config, prep, history, trainer.best_epoch)
    history.to_csv(RUNS_DIR / f'ae_z{z}_history.csv', index=False)
    results[z] = history
    print(f'  saved {ckpt_path.name}  (best epoch {trainer.best_epoch})')

## 4. Compare z_dim

In [ ]:
summary_rows = []
for z, hist in results.items():
    best = hist.loc[hist['val_loss'].idxmin()]
    summary_rows.append({
        'z_dim':      z,
        'best_epoch': int(best['epoch']),
        'val_loss':   round(float(best['val_loss']), 6),
        'val_num':    round(float(best['val_num']),  6),
        'val_cat':    round(float(best['val_cat']),  6),
    })
summary_df = pd.DataFrame(summary_rows).set_index('z_dim')
print(summary_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, len(Z_DIMS), figsize=(15, 4), sharey=False)
for ax, (z, hist) in zip(axes, results.items()):
    ax.plot(hist['epoch'], hist['train_loss'], label='train', linewidth=1.4)
    ax.plot(hist['epoch'], hist['val_loss'],   label='val',   linewidth=1.4)
    best_ep = int(hist.loc[hist['val_loss'].idxmin(), 'epoch'])
    ax.axvline(best_ep, color='gray', linestyle='--', linewidth=0.8, label=f'best={best_ep}')
    ax.set_title(f'z_dim = {z}')
    ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(fontsize=8)
plt.suptitle('Train vs Validation Loss per z_dim', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ae_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Select Best z

**Trade-off**: smaller `z` compresses more (useful for anomaly detection later) but
sacrifices reconstruction quality. `z=16` typically balances both;
 adjust `BEST_Z` after inspecting the table.

In [ ]:
BEST_Z = 16  # adjust after inspecting the summary table

shutil.copy(CHECKPOINTS_DIR / f'ae_z{BEST_Z}.pt', CHECKPOINTS_DIR / 'ae_best.pt')
print(f'Best checkpoint: z_dim={BEST_Z}  ->  checkpoints/ae_best.pt')

## 6. Evaluate on Test Set

In [ ]:
model_best, config_best, prep_best, history_best, best_ep = load_checkpoint(
    CHECKPOINTS_DIR / 'ae_best.pt', device=DEVICE
)
model_best.eval()
evaluator = ReconstructionEvaluator(model_best, prep_best, DEVICE)

metrics  = evaluator.summary_metrics(x_num_test, x_cat_test)
baseline = evaluator.categorical_baseline(x_cat_test)

print(f'MSE numeric: {metrics["mse_numeric"]:.6f}')
print('\nCategorical accuracy — AE vs majority-class baseline:')
print(f'{"column":<30} {"baseline":>10} {"AE":>10} {"delta":>8}')
print('-' * 62)
for col in metrics['cat_accuracy']:
    b = baseline[col]
    a = metrics['cat_accuracy'][col]
    print(f'{col:<30} {b:>10.4f} {a:>10.4f} {a - b:>+8.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_best['epoch'], history_best['train_loss'], label='train', linewidth=1.5)
ax.plot(history_best['epoch'], history_best['val_loss'],   label='val',   linewidth=1.5)
best_ep_val = int(history_best.loc[history_best['val_loss'].idxmin(), 'epoch'])
ax.axvline(best_ep_val, color='gray', linestyle='--', alpha=0.7,
           label=f'best epoch={best_ep_val}')
ax.set_title(f'Learning curve — ae_best (z_dim={BEST_Z})')
ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'ae_z{BEST_Z}_curve.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Final val_loss={history_best["val_loss"].iloc[-1]:.6f}')

In [ ]:
feat_df = evaluator.per_feature_metrics(x_num_test, x_cat_test)
print('Top 10 hardest features (highest MSE):')
print(feat_df.sort_values('mse', ascending=False).head(10).to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
top20 = feat_df.nlargest(20, 'mse')
ax1.barh(top20['feature'][::-1], top20['mse'][::-1], color='steelblue')
ax1.set_xlabel('MSE (standardised)'); ax1.set_title('Top 20 features by MSE')
ax2.scatter(feat_df['pearson_corr'], feat_df['mse'], alpha=0.5, s=20)
ax2.set_xlabel('Pearson corr (orig vs recon)'); ax2.set_ylabel('MSE')
ax2.set_title('MSE vs correlation per feature')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ae_per_feature_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
errors   = evaluator.per_sample_error(x_num_test, x_cat_test)
class_df = evaluator.metrics_by_class(errors, y_test)

print('=== Reconstruction error by class (ae_best, z={}) ==='.format(BEST_Z))
print(class_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
for label, name, color in [(0, 'normal', 'steelblue'), (1, 'churn', 'tomato')]:
    ax.hist(errors[y_test == label], bins=100, alpha=0.55,
            label=f'{name} (n={int((y_test==label).sum()):,})',
            color=color, density=True)
ax.set_xlabel('Per-sample reconstruction error')
ax.set_title(f'Error distribution: churn vs normal (z={BEST_Z})')
ax.legend(); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ae_error_by_class.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(x_num_test), size=8, replace=False)

panel = evaluator.sample_panel(
    x_num_test[sample_idx],
    x_cat_test[sample_idx],
    df.iloc[split['test']].iloc[sample_idx].reset_index(drop=True),
)
cols_preview = prep_best.feature_names[:15]
print('Original vs Reconstructed (first 15 features, transposed):')
print(panel[cols_preview].T.to_string())

In [ ]:
diff = evaluator.difference_matrix(x_num_test[sample_idx], x_cat_test[sample_idx])

fig, ax = plt.subplots(figsize=(20, 3))
im = ax.imshow(diff, aspect='auto', cmap='Reds')
ax.set_xlabel(f'Non-constant numeric feature (0–{prep_best.numeric_dim - 1})')
ax.set_ylabel('Client'); ax.set_yticks(range(8))
ax.set_title('|original − reconstructed| per client × feature (standardised)')
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ae_difference_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Analysis

**Fill in after running:**

- **Constant columns**: check `prep.n_const` — these features are excluded from the model.
- **Which z_dim and why?** `z=16` balances compression vs reconstruction quality.
- **Learning curve (z=16)**: verify the val curve is flat near the best epoch — if still dropping, increase `max_epochs`.
- **Categorical baseline vs AE**: the AE should exceed the majority-class baseline (otherwise it learns nothing useful for categoricals).
- **Churn vs normal error gap?** Churned clients tend to reconstruct worse because the AE learned only normal behaviour — foundation for anomaly-based detection.
- **Hardest features?** Typically sparse / highly-skewed listening-event features.
- **Heatmap observation?** A small subset of features drives most of the loss.
